# Fase 3 — Preparação dos Dados (CRISP-DM)

**Projeto:** Análise dos casos de violência doméstica no Distrito Federal
**Pergunta de negócio:** Quais regiões administrativas do Distrito Federal apresentam os maiores registros de violência doméstica de 2024 a 2025?

Os dados oficiais da SSP-DF só são publicados como **tabelas dentro de relatórios em PDF**. Este notebook faz a extração dessas tabelas e monta uma **planilha estruturada** (CSV e Excel), seguindo as quatro tarefas da Fase 3 do guia CRISP-DM:

| Tarefa | O que é feito aqui |
|---|---|
| **Selecionar** (filtrar) | Escolher os relatórios e a tabela de interesse (Tabela 1 — crimes por RA) |
| **Limpar** (tratar) | Ler o PDF (texto ou OCR), corrigir erros de leitura, padronizar nomes de RA |
| **Construir** (derivar) | Criar participação %, variação, ranking e indicador de comparabilidade |
| **Integrar** (mesclar) | Juntar todos os relatórios em uma única base, ficando com a revisão mais recente de cada ano |

## 0. Ambiente e reprodução

In [1]:
import sys, platform
from pathlib import Path
import pandas as pd

RAIZ = Path('..').resolve()          # pasta Projeto_de_Ciência_de_Dados
sys.path.insert(0, str(RAIZ / 'src'))
import extracao

BRUTOS = RAIZ / 'dados' / 'brutos'
PROCESSADOS = RAIZ / 'dados' / 'processados'

import pdfplumber, pytesseract
print('Python', platform.python_version(), '| pandas', pd.__version__, '| pdfplumber', pdfplumber.__version__,
      '| Tesseract', pytesseract.get_tesseract_version())

Python 3.13.18 | pandas 3.0.5 | pdfplumber 0.11.10 | Tesseract 5.3.4


## 3.1 Selecionar — relatórios utilizados

Todos são documentos técnicos da SSP-DF (Coordenação de Análise de Fenômenos de Segurança Pública), com a mesma Tabela 1 por Região Administrativa. Cada relatório compara dois períodos, o que permite montar uma série e conferir revisões.

O relatório **RAC-001/2026** é a fonte principal (anos 2024 e 2025). Os demais servem para contexto histórico e para validar a consistência da extração.

In [2]:
pdfs = sorted(BRUTOS.glob('*.pdf'))
pd.DataFrame({'arquivo': [p.name for p in pdfs], 'tamanho_kb': [round(p.stat().st_size/1024) for p in pdfs]})

,arquivo,tamanho_kb
0,SSPDF_FSP-002-2021_VD_2019-2020.pdf,1289
1,SSPDF_FSP-003-2023_VD_2021-2022.pdf,1659
2,SSPDF_FSP-005-2020_VD_2018-2019.pdf,1195
3,SSPDF_RAC-001-2026_VD_2024-2025.pdf,1492
4,SSPDF_RAC-020-2026_VD_1sem2025-2026.pdf,1442


## 3.2 Limpar — extração das tabelas

O módulo `src/extracao.py` faz o trabalho pesado:

1. tenta ler a página 1 como **texto** (`pdfplumber`). Funciona nos relatórios de 2020–2023;
2. se a tabela for uma **imagem** (relatórios RAC de 2026), renderiza a página e aplica **OCR** com o Tesseract em português;
3. corrige erros típicos de OCR em números (`$1` → `51`, `O` → `0`);
4. padroniza os nomes das RAs (`SOBRADINHO 2`, `Sobradinho 2` → `Sobradinho II`; `Sol Nascente/Por Do Sol` → `Sol Nascente/Pôr do Sol`).

Cada linha extraída é **validada duas vezes**: a diferença entre os dois anos precisa bater com a coluna *Quantit.* do próprio relatório, e a soma das RAs precisa bater com a linha *TOTAL*.

In [3]:
tabelas, longa, base, log, revisoes = extracao.main(BRUTOS, PROCESSADOS)

                              documento       ra  tipo                                                  mensagem
    SSPDF_FSP-002-2021_VD_2019-2020.pdf    TOTAL    ok 2019: soma das 33 RAs = 16861; TOTAL do relatório = 16861
    SSPDF_FSP-002-2021_VD_2019-2020.pdf    TOTAL    ok 2020: soma das 33 RAs = 15995; TOTAL do relatório = 15995
    SSPDF_FSP-003-2023_VD_2021-2022.pdf    TOTAL    ok 2021: soma das 33 RAs = 16791; TOTAL do relatório = 16791
    SSPDF_FSP-003-2023_VD_2021-2022.pdf    TOTAL    ok 2022: soma das 33 RAs = 16949; TOTAL do relatório = 16949
    SSPDF_FSP-005-2020_VD_2018-2019.pdf    TOTAL    ok 2018: soma das 31 RAs = 15368; TOTAL do relatório = 15368
    SSPDF_FSP-005-2020_VD_2018-2019.pdf    TOTAL    ok 2019: soma das 31 RAs = 16549; TOTAL do relatório = 16549
    SSPDF_RAC-001-2026_VD_2024-2025.pdf Park Way aviso   coluna 'Quantit.' ilegível ('[)'); conferida pelo total
    SSPDF_RAC-001-2026_VD_2024-2025.pdf    TOTAL    ok 2024: soma das 33 RAs = 21721; TOTAL do r

### Resumo da validação

In [4]:
erros = log[log['tipo'] == 'ERRO']
print(f"{len(tabelas)} relatórios | {len(longa)} valores extraídos | {len(erros)} erros de validação")
assert erros.empty, 'Há divergências: conferir manualmente antes de seguir'
log[log['ra'] == 'TOTAL']

5 relatórios | 326 valores extraídos | 0 erros de validação


,documento,ra,tipo,mensagem
0,SSPDF_FSP-002-2021_VD_2019-2020.pdf,TOTAL,ok,2019: soma das 33 RAs = 16861; TOTAL do relató...
1,SSPDF_FSP-002-2021_VD_2019-2020.pdf,TOTAL,ok,2020: soma das 33 RAs = 15995; TOTAL do relató...
2,SSPDF_FSP-003-2023_VD_2021-2022.pdf,TOTAL,ok,2021: soma das 33 RAs = 16791; TOTAL do relató...
3,SSPDF_FSP-003-2023_VD_2021-2022.pdf,TOTAL,ok,2022: soma das 33 RAs = 16949; TOTAL do relató...
4,SSPDF_FSP-005-2020_VD_2018-2019.pdf,TOTAL,ok,2018: soma das 31 RAs = 15368; TOTAL do relató...
5,SSPDF_FSP-005-2020_VD_2018-2019.pdf,TOTAL,ok,2019: soma das 31 RAs = 16549; TOTAL do relató...
7,SSPDF_RAC-001-2026_VD_2024-2025.pdf,TOTAL,ok,2024: soma das 33 RAs = 21721; TOTAL do relató...
8,SSPDF_RAC-001-2026_VD_2024-2025.pdf,TOTAL,ok,2025: soma das 33 RAs = 23066; TOTAL do relató...
9,SSPDF_RAC-020-2026_VD_1sem2025-2026.pdf,TOTAL,ok,2025: soma das 33 RAs = 11709; TOTAL do relató...
10,SSPDF_RAC-020-2026_VD_1sem2025-2026.pdf,TOTAL,ok,2026: soma das 33 RAs = 11206; TOTAL do relató...


### Padronização dos nomes das RAs

Variações de escrita encontradas nos relatórios e o nome padronizado adotado:

In [5]:
nomes = (longa[['ra_original', 'ra']].drop_duplicates()
         .groupby('ra')['ra_original'].apply(lambda s: ' | '.join(sorted(set(s)))).reset_index())
nomes.columns = ['RA padronizada', 'Como aparece nos PDFs']
nomes

,RA padronizada,Como aparece nos PDFs
0,Arniqueira,ARNIQUEIRA | Arniqueira
1,Brasília,BRASILIA | Brasilia
2,Brazlândia,BRAZLANDIA | Brazlandia
3,Candangolândia,CANDANGOLANDIA | Candangolandia
4,Ceilândia,CEILANDIA | Ceilandia
5,Cruzeiro,CRUZEIRO | Cruzeiro
6,Estrutural,ESTRUTURAL | Estrutural
7,Fercal,FERCAL | Fercal
8,Gama,GAMA | Gama
9,Guará,GUARA | Guara


### Revisões entre relatórios

A SSP conta os registros **pela data do fato** e avisa que os números estão "sujeitos a alterações". Por isso o mesmo ano aparece com valores diferentes em dois relatórios. Exemplo: 2019 no relatório de jan/2020 e no de jan/2021.

Além disso, em 2019 foram criadas as RAs **Sol Nascente/Pôr do Sol** (desmembrada de Ceilândia) e **Arniqueira** (desmembrada de Águas Claras). O relatório de 2020 ainda usa 31 RAs; o de 2021 já reclassifica 2019 em 33 RAs.

In [6]:
totais = longa.groupby(['documento', 'periodo', 'ano'])['registros'].agg(['sum', 'count']).reset_index()
totais.columns = ['documento', 'periodo', 'ano', 'total', 'n_RAs']
totais

,documento,periodo,ano,total,n_RAs
0,SSPDF_FSP-002-2021_VD_2019-2020.pdf,Anual,2019,16861,33
1,SSPDF_FSP-002-2021_VD_2019-2020.pdf,Anual,2020,15995,33
2,SSPDF_FSP-003-2023_VD_2021-2022.pdf,Anual,2021,16791,33
3,SSPDF_FSP-003-2023_VD_2021-2022.pdf,Anual,2022,16949,33
4,SSPDF_FSP-005-2020_VD_2018-2019.pdf,Anual,2018,15368,31
5,SSPDF_FSP-005-2020_VD_2018-2019.pdf,Anual,2019,16549,31
6,SSPDF_RAC-001-2026_VD_2024-2025.pdf,Anual,2024,21721,33
7,SSPDF_RAC-001-2026_VD_2024-2025.pdf,Anual,2025,23066,33
8,SSPDF_RAC-020-2026_VD_1sem2025-2026.pdf,1º semestre,2025,11709,33
9,SSPDF_RAC-020-2026_VD_1sem2025-2026.pdf,1º semestre,2026,11206,33


In [7]:
if not revisoes.empty:
    rev = revisoes.copy()
    docs = [c for c in rev.columns if str(c).endswith('.pdf')]
    rev['diferença'] = rev[docs[-1]] - rev[docs[0]]
    display(rev.sort_values('diferença', key=abs, ascending=False).head(10))

documento,periodo,ano,ra,SSPDF_FSP-002-2021_VD_2019-2020.pdf,SSPDF_FSP-005-2020_VD_2018-2019.pdf,diferença
3,Anual,2019,Ceilândia,2161.0,2816.0,655.0
30,Anual,2019,Águas Claras,289.0,517.0,228.0
0,Anual,2019,Brasília,816.0,769.0,-47.0
16,Anual,2019,Planaltina,1446.0,1423.0,-23.0
27,Anual,2019,Taguatinga,1002.0,980.0,-22.0
7,Anual,2019,Gama,1004.0,986.0,-18.0
18,Anual,2019,Riacho Fundo,285.0,268.0,-17.0
17,Anual,2019,Recanto das Emas,922.0,907.0,-15.0
23,Anual,2019,Sobradinho,552.0,538.0,-14.0
29,Anual,2019,Vicente Pires,374.0,360.0,-14.0


**Regra adotada:** quando um ano aparece em mais de um relatório, vale o relatório com a **data de atualização mais recente** (`extracao.consolidar`). O ano de 2018 só existe no formato de 31 RAs: Ceilândia e Águas Claras de 2018 ainda incluem Sol Nascente e Arniqueira. Por isso 2018 recebe a marcação `comparavel_33_RAs = False`.

## 3.3 Construir — variáveis derivadas

In [8]:
base = base.copy()
base['total_periodo'] = base.groupby(['periodo', 'ano'])['registros'].transform('sum')
base['participacao_pct'] = (100 * base['registros'] / base['total_periodo']).round(2)
base['ranking'] = base.groupby(['periodo', 'ano'])['registros'].rank(ascending=False, method='min').astype(int)
base['n_ras_no_periodo'] = base.groupby(['periodo', 'ano'])['ra'].transform('count')
base['comparavel_33_RAs'] = base['n_ras_no_periodo'] == 33

anual = base[base['periodo'] == 'Anual'].pivot(index='ra', columns='ano', values='registros')
anual.columns = [f'reg_{c}' for c in anual.columns]
semestral = base[base['periodo'] == '1º semestre'].pivot(index='ra', columns='ano', values='registros')
semestral.columns = [f'reg_1sem_{c}' for c in semestral.columns]
larga = anual.join(semestral, how='outer')

if {'reg_2024', 'reg_2025'} <= set(larga.columns):
    larga['var_abs_2024_2025'] = larga['reg_2025'] - larga['reg_2024']
    larga['var_pct_2024_2025'] = (100 * larga['var_abs_2024_2025'] / larga['reg_2024']).round(1)
    larga['ranking_2025'] = larga['reg_2025'].rank(ascending=False, method='min').astype(int)
    larga = larga.sort_values('ranking_2025')
larga.head(12)

,reg_2018,reg_2019,reg_2020,reg_2021,reg_2022,reg_2024,reg_2025,reg_1sem_2025,reg_1sem_2026,var_abs_2024_2025,var_pct_2024_2025,ranking_2025
ra,,,,,,,,,,,,
Ceilândia,2479.0,2161.0,2220.0,2126.0,2112.0,2922.0,3135.0,1613,1542,213.0,7.3,1
Planaltina,1234.0,1446.0,1321.0,1313.0,1467.0,1702.0,1951.0,974,988,249.0,14.6,2
Samambaia,1167.0,1392.0,1262.0,1278.0,1359.0,1691.0,1809.0,900,885,118.0,7.0,3
Taguatinga,1000.0,1002.0,970.0,1043.0,960.0,1277.0,1339.0,699,601,62.0,4.9,4
Recanto das Emas,918.0,922.0,847.0,950.0,880.0,1156.0,1303.0,669,579,147.0,12.7,5
Gama,892.0,1004.0,883.0,852.0,820.0,1158.0,1238.0,673,600,80.0,6.9,6
Sol Nascente/Pôr do Sol,NaN,702.0,765.0,804.0,813.0,1093.0,1233.0,638,658,140.0,12.8,7
Santa Maria,731.0,826.0,767.0,830.0,804.0,1145.0,1198.0,604,528,53.0,4.6,8
São Sebastião,719.0,794.0,738.0,857.0,810.0,1031.0,1086.0,544,544,55.0,5.3,9


## 3.4 Integrar — base final e dicionário de dados

In [9]:
dicionario = pd.DataFrame([
    ('documento', 'Arquivo PDF de origem (SSP-DF)'),
    ('metodo_extracao', 'texto (pdfplumber) ou OCR (Tesseract)'),
    ('data_atualizacao', 'Data em que a SSP extraiu os dados do Banco Millenium'),
    ('periodo', 'Anual (jan–dez) ou 1º semestre (jan–jun)'),
    ('ano', 'Ano do fato'),
    ('ra_original', 'Nome da RA como escrito no PDF'),
    ('ra', 'Nome padronizado da Região Administrativa'),
    ('registros', 'Ocorrências de violência doméstica ou familiar (Lei 11.340/2006) registradas'),
    ('total_periodo', 'Total do DF no período (soma das RAs)'),
    ('participacao_pct', 'Participação da RA no total do DF (%)'),
    ('ranking', 'Posição da RA no período (1 = mais registros)'),
    ('n_ras_no_periodo', 'Quantidade de RAs na tabela do período (31 ou 33)'),
    ('comparavel_33_RAs', 'True se o período usa a divisão atual de 33 RAs'),
], columns=['variável', 'descrição'])

base.to_csv(PROCESSADOS / 'violencia_domestica_ra.csv', index=False, encoding='utf-8-sig')
with pd.ExcelWriter(PROCESSADOS / 'violencia_domestica_ra.xlsx') as xls:
    larga.reset_index().to_excel(xls, sheet_name='por_RA_e_ano', index=False)
    base.to_excel(xls, sheet_name='base_longa', index=False)
    log.to_excel(xls, sheet_name='log_validacao', index=False)
    totais.to_excel(xls, sheet_name='totais_por_relatorio', index=False)
    dicionario.to_excel(xls, sheet_name='dicionario', index=False)
print('Arquivos gerados em', PROCESSADOS.relative_to(RAIZ))
for f in sorted(PROCESSADOS.iterdir()):
    print(' -', f.name)

Arquivos gerados em dados/processados
 - extracao_bruta.csv
 - log_validacao.csv
 - violencia_domestica_ra.csv
 - violencia_domestica_ra.xlsx


In [10]:
dicionario

,variável,descrição
0,documento,Arquivo PDF de origem (SSP-DF)
1,metodo_extracao,texto (pdfplumber) ou OCR (Tesseract)
2,data_atualizacao,Data em que a SSP extraiu os dados do Banco Mi...
3,periodo,Anual (jan–dez) ou 1º semestre (jan–jun)
4,ano,Ano do fato
5,ra_original,Nome da RA como escrito no PDF
6,ra,Nome padronizado da Região Administrativa
7,registros,Ocorrências de violência doméstica ou familiar...
8,total_periodo,Total do DF no período (soma das RAs)
9,participacao_pct,Participação da RA no total do DF (%)


## Conclusão da Fase 3

- Os relatórios foram convertidos em uma **planilha estruturada** (`dados/processados/violencia_domestica_ra.xlsx` e `.csv`), com uma linha por RA, período e ano.
- **Todas as somas extraídas conferem com os totais oficiais** dos relatórios, inclusive nos PDFs lidos por OCR.
- Os nomes das RAs foram padronizados e as revisões entre relatórios foram documentadas.
- A base está pronta para a **Fase 4 (Modelagem)**, que responde à pergunta de negócio com os dados de 2024 e 2025.